# ADR Detector — Fine-Tuning Notebook

This notebook fine-tunes a biomedical BERT model on the **CADEC** dataset so it learns to recognize
drugs (`Drug`) and adverse drug reactions (`ADR`) the way people actually write about them in informal,
casual text — instead of the formal clinical-note style the original pretrained model was trained on.

**Before running anything:** go to `Runtime > Change runtime type` and select a **T4 GPU**. Fine-tuning
on CPU would take hours instead of minutes.

**What this notebook does, step by step:**
1. Install libraries and upload your CADEC data
2. Parse the raw text + annotation files into a table (same logic as `explore_data.py`)
3. Convert each post into token-level labels the model can actually learn from
4. Fine-tune a pretrained biomedical BERT model on those labels
5. Evaluate it and try it on an example sentence
6. Upload your fine-tuned model to the Hugging Face Hub (free) so `app.py` can use it


## Step 1: Install libraries


In [1]:
!pip install -q -U transformers datasets evaluate seqeval accelerate huggingface_hub



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Upload your CADEC data

Run the cell below, then click **Choose Files** and select `CADEC.v2.zip` from your computer
(the same file you downloaded from `data.csiro.au` earlier).


In [ ]:
from google.colab import files

uploaded = files.upload()  # select CADEC.v2.zip when prompted


In [ ]:
import zipfile

with zipfile.ZipFile("CADEC.v2.zip", "r") as z:
    z.extractall("data")

print("Extracted.")


## Step 3: Parse the data

Same parsing logic as `explore_data.py` in your project — reads each post's raw text and its matching
`.ann` annotation file (drug/ADR/disease/symptom spans), and builds one row per entity.


In [ ]:
import pandas as pd
from pathlib import Path

TEXT_DIR = Path("data/cadec/text")
ANN_DIR = Path("data/cadec/original")


def parse_ann_file(ann_path: Path):
    """Parse a single brat-format .ann file into a list of entity dicts."""
    entities = []
    for line in ann_path.read_text(encoding="utf-8").splitlines():
        if not line.startswith("T"):
            continue  # skip AnnotatorNotes (#...) and relation lines
        parts = line.split("\t")
        if len(parts) < 3:
            continue
        entity_id, label_span, text = parts[0], parts[1], parts[2]
        label, *span_tokens = label_span.split(" ")
        span_str = " ".join(span_tokens)

        # Some entities are "discontinuous" (e.g. "9 19;25 30"). We keep it
        # simple and take the outer start/end of the whole span.
        spans = [s.split() for s in span_str.split(";")]
        start = int(spans[0][0])
        end = int(spans[-1][1])

        entities.append(
            {"entity_id": entity_id, "label": label, "start": start, "end": end, "text": text}
        )
    return entities


def load_cadec():
    rows = []
    for text_path in sorted(TEXT_DIR.glob("*.txt")):
        ann_path = ANN_DIR / (text_path.stem + ".ann")
        if not ann_path.exists():
            continue
        post_text = text_path.read_text(encoding="utf-8")
        for entity in parse_ann_file(ann_path):
            rows.append(
                {
                    "post_id": text_path.stem,
                    "label": entity["label"],
                    "entity_text": entity["text"],
                    "start": entity["start"],
                    "end": entity["end"],
                    "post_text": post_text,
                }
            )
    return pd.DataFrame(rows)


entities_df = load_cadec()
print(f"Loaded {len(entities_df)} entity annotations from {entities_df.post_id.nunique()} posts")
entities_df.head()


## Step 4: Convert to token-level BIO labels

Models don't understand "characters 9 to 19 are an ADR" — they need a label attached to each *token*
(roughly, each word/sub-word piece). We use the **BIO scheme**:
- `B-ADR` = the first token of an ADR mention
- `I-ADR` = a token that continues an ADR mention already in progress
- `O` = not part of any entity
(same pattern for `Drug`, `Disease`, `Symptom`, `Finding`)

We only keep `Drug` and `ADR` as the labels we actually care about for this project — those map directly
to "drug mentioned" and "adverse reaction mentioned" in the app.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ENTITY_TYPES = ["ADR", "Drug"]  # the only two we're training on
PRIORITY = ["ADR", "Drug"]      # if annotations overlap, ADR wins

label_list = ["O"] + [f"{p}-{e}" for e in ENTITY_TYPES for p in ["B", "I"]]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
label_list


In [ ]:
def tokenize_and_align_labels(text, entities):
    """Tokenize one post and assign a B-/I-/O label to each token using
    the entity's character start/end position (an "offset").
    """
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=256)
    offsets = encoding.pop("offset_mapping")

    label_ids = []
    is_special = []
    for start, end in offsets:
        if start == end:
            label_ids.append(-100)  # special token, e.g. [CLS]/[SEP] -- ignored during training
            is_special.append(True)
        else:
            label_ids.append(label2id["O"])
            is_special.append(False)

    relevant_entities = [e for e in entities if e["label"] in ENTITY_TYPES]
    sorted_entities = sorted(relevant_entities, key=lambda e: PRIORITY.index(e["label"]))

    claimed = [False] * len(offsets)
    for ent in sorted_entities:
        first = True
        for i, (start, end) in enumerate(offsets):
            if is_special[i] or claimed[i]:
                continue
            if start >= ent["start"] and end <= ent["end"]:
                tag = ("B-" if first else "I-") + ent["label"]
                label_ids[i] = label2id[tag]
                claimed[i] = True
                first = False

    encoding["labels"] = label_ids
    return encoding


In [ ]:
# Group entities by post, so each training example is one whole forum post
post_texts = {p.stem: p.read_text(encoding="utf-8") for p in TEXT_DIR.glob("*.txt")}
entities_by_post = {
    pid: group.to_dict("records")
    for pid, group in entities_df.groupby("post_id")
}

examples = []
for post_id, text in post_texts.items():
    entities = entities_by_post.get(post_id, [])
    examples.append(tokenize_and_align_labels(text, entities))

print(f"Built {len(examples)} training examples")


## Step 5: Train / validation split

We hold out 15% of posts to check how well the model does on posts it never trained on.


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_examples, val_examples = train_test_split(examples, test_size=0.15, random_state=42)

train_ds = Dataset.from_list(train_examples)
val_ds = Dataset.from_list(val_examples)

print(f"Train: {len(train_ds)}  |  Validation: {len(val_ds)}")


## Step 6: Load the pretrained model and fine-tune it


In [ ]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer)


In [ ]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="adr-ner-cadec",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Roughly: with ~1,200 posts and 5 epochs on a free T4 GPU, this should take somewhere around 10-20 minutes. If it's taking dramatically longer, double check `Runtime > Change runtime type` actually has a GPU selected.


## Step 7: Evaluate


In [ ]:
trainer.evaluate()


## Step 8: Try it on an example

Same sentence we tested the pretrained model on earlier — see how the fine-tuned version does.


In [ ]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,  # use the GPU
    aggregation_strategy="simple",
)

ner("started metformin last week and I can't stop feeling nauseous")


## Step 9: Upload your fine-tuned model to the Hugging Face Hub

1. Create a free account at [huggingface.co](https://huggingface.co) if you don't have one
2. Go to **Settings > Access Tokens > New token**, create one with **write** permission
3. In Colab, click the **key icon** in the left sidebar, add a new secret named `HF_TOKEN`, paste your
   token as the value, and toggle notebook access **on**
4. Change `HF_REPO` below to `your-username/adr-ner-cadec`


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))


In [ ]:
HF_REPO = "your-username/adr-ner-cadec"  # <-- change this to your Hugging Face username

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)


## Step 10: Use this model in your app

Back in `app.py`, two changes:

1. Change the model name:
```python
    model="your-username/adr-ner-cadec",
```
2. Update the label names `extract_entities()` looks for, since this model uses different label names
   than the general pretrained one:
```python
    drugs_found = [word for label, word in merged if label == "Drug"]
    reactions_found = [word for label, word in merged if label == "ADR"]
```
